# Avance 5: ensambles y modelo final para estimar la emisión de NOₓ

Equipo 39: Isabella Arango Restrepo (A01796305), Isaí Ambrocio (A01625101) y Manuel Alejandro Vázquez Meza (A01796404).

## 1. Objetivo

Combinar los modelos del avance 4 en ensambles que aprovechen sus fortalezas, compararlos con el mejor modelo individual con el mismo banco de evaluación (noviembre–diciembre de 2023, una estación oculta a la vez, placebo del domingo), elegir el modelo final y evaluarlo una sola vez en datos que ningún modelo vio: febrero–mayo de 2024 y las contingencias ambientales de ese periodo.

Reglas del ejercicio:

- Todo ensamble sigue estimando emisión. Como la concentración es lineal en la emisión, una combinación convexa (pesos ≥ 0 que suman 1) de las predicciones corresponde a la misma combinación de las emisiones.
- La elección del modelo final usa solo noviembre–diciembre de 2023. Febrero–mayo de 2024 se usa una vez, con la configuración congelada (`nox-ml evaluate`).
- Todo el código de este cuaderno llama a `nox_ml`; los números salen de `results/runs/` y, para las figuras del modelo final, de los datos observados del periodo de cada corrida.

In [ ]:
from nox_ml.validation import report, report_final

RUNS = report.find_runs("results/runs")
ROOT = RUNS.parents[1]
FINAL = "pinn_puff"  # [POR DECIDIR] corrida del modelo final elegido en la sección 5
print(f"Corridas en {RUNS.relative_to(ROOT)}; modelo final provisional: {FINAL}")

## 2. Los ensambles

| # | Ensamble | Tipo | Modelos base | Qué combina | Hiperparámetros |
|---|---|---|---|---|---|
| X1 | `bagged_inversion` | Homogéneo (bagging) | E5b `pinn_puff` | Inversiones sobre remuestreos de días completos | número de remuestreos, tamaño del bloque |
| X2 | `wind_ensemble` | Homogéneo | E2 `inversion_puff` | Inversiones con el viento y la capa límite perturbados | número de miembros, desviación de la dirección, de la velocidad y de la capa límite |
| X3 | `pinn_seeds` (opcional) | Homogéneo | E5b `pinn_puff` | Redes de la PINN con semillas distintas | número de semillas |
| X4 | `gb_deweather` | Homogéneo (boosting) | E6 `rf_deweather` | Gradient boosting en lugar de bosque aleatorio | tasa de aprendizaje, iteraciones, hojas, regularización |
| X5 | `stacking` | Heterogéneo (stacking) | E5b, E2, E5a | Pesos convexos aprendidos sobre las predicciones fuera de muestra, sin la estación evaluada | modelos base, regla |
| X6 | `bayes_average` | Heterogéneo | E5b, E2, E5a | Pesos por verosimilitud en estación oculta | modelos base, temperatura |

[POR ESCRIBIR] Un párrafo por ensamble: por qué se esperaba que mejorara, qué debilidad del modelo base ataca y si su emisión es absoluta o relativa (las bases con transportes distintos solo comparten el cociente por tipo de día).

In [ ]:
table = report_final.comparison(RUNS)
pending = report_final.missing(table)
print(
    "Todos los modelos y ensambles previstos tienen corrida." if pending.empty else f"Faltan {len(pending)}:"
)
pending

## 3. Ajuste de hiperparámetros

Los ensambles con hiperparámetros propios se ajustan con `nox-ml tune` en noviembre–diciembre de 2023 (X1, X2, X4). Los pesos de X5 y X6 se aprenden con `nox_ml.validation.stacking`, que no reajusta los modelos base: la búsqueda es la elección de las bases y de la regla. Como en el avance 4, el ajuste no es anidado y su error es algo optimista; el error sin sesgo es el del periodo de prueba (sección 6).

In [ ]:
trials = report.read_trials(RUNS)
searched = {
    run: t
    for run, t in trials.items()
    if report_final.config_model(RUNS / run) not in ("pinn_puff", "inversion_puff")
}
print("Búsquedas de ensambles:", ", ".join(searched) or "ninguna todavía")
for run, t in searched.items():
    print(f"\n{run}: {len(t)} pruebas, de {t['loo_rmse_ppb'].min():.2f} a {t['loo_rmse_ppb'].max():.2f} ppb")
    print(report.search_space(t).to_string(index=False))

[POR ESCRIBIR] Por búsqueda: espacio explorado, número de pruebas, mejor configuración y mejora frente a la configuración inicial, leída contra el ruido de ±0.5 ppb.

## 4. Tabla comparativa

El mejor modelo individual del avance 4 (E5b) y los ensambles, ordenados por la métrica principal: el error al predecir una estación oculta (`loo_rmse_ppb`). Para X5 y X6 el error es el anidado (`loo_source` = anidado): cada estación se predice con pesos aprendidos sin ella. Las demás métricas: domingo/hábil frente al tráfico contado (0.65–0.73), placebo (elegible si su centro queda en 1 ± 0.15), sesgo y error de hora en hora, y el tiempo de entrenamiento.

In [ ]:
observed_tuning = report_final.observed_or_none(RUNS / FINAL, ROOT)
table = report_final.comparison(RUNS, observed_tuning)
print(report_final.kinds_covered(table).to_string())
columns = [
    "id",
    "run",
    "kind",
    "loo_source",
    "loo_rmse_ppb",
    "loo_bias_ppb",
    "loo_hourly_rmse_ppb",
    "sunday_ratio",
    "sunday_ratio_gap",
    "placebo_center",
    "placebo_halfwidth",
    "eligible",
    "fit_seconds",
    "n_params",
]
table[[c for c in columns if c in table.columns]].round(2)

[POR ESCRIBIR] Lectura de la tabla: qué ensambles mejoran al mejor individual más allá del ruido, cuáles solo mejoran el placebo o el domingo, y qué cuestan.

## 5. Modelo final

Criterios, fijados antes de ver la tabla:

1. Pasa el placebo (centro en 1 ± 0.15).
2. Menor error con estación oculta.
3. Si dos candidatos quedan a menos de 0.5 ppb (el ruido), se prefiere el que da emisión absoluta por zona, después el de placebo más estrecho y después el más barato.

[POR ESCRIBIR] Decisión y argumentos más allá de la métrica: interpretabilidad (emisión por zona), incertidumbre, emisión absoluta o relativa, costo y riesgos.

## 6. Evaluación en datos no vistos

El modelo final y los finalistas se corren una sola vez en el periodo de prueba (15 de febrero–31 de mayo de 2024, los días con píxeles de TEMPO) con la configuración congelada: `nox-ml evaluate runs/<nombre>/config.yaml --period test`. Ningún modelo ni ensamble se ajustó ni se eligió con estos datos.

In [ ]:
test = report_final.test_period(RUNS)
print(
    "Sin corridas en el periodo de prueba todavía."
    if test.empty
    else f"{len(test)} corridas en el periodo de prueba"
)
test[[c for c in columns if c in test.columns]].round(2) if not test.empty else test

[POR ESCRIBIR] Comparación entre noviembre–diciembre de 2023 y febrero–mayo de 2024: si el error y el placebo se sostienen, y qué cambia con la temporada (más ozono, más ventilación).

### Contingencias ambientales de 2024

[POR ESCRIBIR] Resultado con el modelo final: cociente de la emisión en los días con medidas (del día siguiente a la activación al de la suspensión, 16 días en 11 episodios) frente a días hábiles comparables, con su placebo de días falsos y el estimador normalizado como referencia.

## 7. Gráficos del modelo final

Las figuras usan las predicciones fuera de muestra del modelo final: cada estación predicha por el modelo ajustado sin ella. Se muestran en el periodo de prueba cuando existe su corrida, y si no, en noviembre–diciembre de 2023.

In [ ]:
final_run = RUNS / f"{FINAL}.test" if (RUNS / f"{FINAL}.test" / "oof.parquet").is_file() else RUNS / FINAL
observed_final = report_final.observed_or_none(final_run, ROOT)
held_out = None if observed_final is None else report_final.held_out(final_run, observed_final)
print(f"Figuras de {final_run.name}" + ("" if held_out is not None else ": sin datos, se omiten"))

### 7.1 Observado frente a predicho en estaciones ocultas

In [ ]:
report_final.plot_observed_vs_predicted(held_out) if held_out is not None else None

[POR ESCRIBIR] Interpretación.

### 7.2 Residuos por estación, por hora y frente a la ventilación

In [ ]:
report_final.plot_residuals(held_out) if held_out is not None else None

[POR ESCRIBIR] Interpretación.

### 7.3 Tendencia y predicción en estaciones ocultas

In [ ]:
report_final.plot_series(held_out, ["MER", "UIZ", "TLA"]) if held_out is not None else None

[POR ESCRIBIR] Interpretación.

### 7.4 Emisión estimada por región

In [ ]:
report_final.plot_emission_map(final_run) if (final_run / "emissions.parquet").is_file() else None

[POR ESCRIBIR] Interpretación.

### 7.5 Cambios de emisión: sábado, domingo y placebo

In [ ]:
periods = {"nov–dic 2023": RUNS / FINAL, "feb–may 2024": RUNS / f"{FINAL}.test"}
report_final.plot_day_types({k: v for k, v in periods.items() if (v / "metrics.json").is_file()})

[POR ESCRIBIR] Interpretación.

### 7.6 Pesos del ensamble

In [ ]:
weights = report_final.ensemble_weights(RUNS / "stacking")
report_final.plot_weights(weights) if weights is not None else print("Sin pesos de stacking todavía.")

[POR ESCRIBIR] Interpretación.

## 8. Limitaciones

[POR ESCRIBIR]

- La subestimación del domingo/hábil frente al tráfico contado (avance 4).
- La capa límite de 2024 es MERRA-2 corregido por hora (ERA5 no cubre enero–junio de 2024). Con la marca horaria corregida (PR #52), la corrección por hora reduce su error mediano fuera de muestra frente a ERA5 (2023) de 383 a 340 m, y la correlación de la tarde es 0.76. En noviembre–diciembre de 2023, escalar la capa límite ±20 % mueve el error de E5b de 11.71 a 12.05 ppb y no cambia el domingo/hábil ni el placebo (`context/knowledge/capa_limite_2024.md`).
- El ajuste no anidado y la fuga residual del stacking anidado (los modelos base vieron la estación evaluada al predecir las demás).
- El nivel absoluto de emisión depende del transporte; los ensambles heterogéneos dan solo cocientes.
- El confinamiento de 2020 queda fuera de esta entrega: el modelo final necesita TEMPO, que no existía.

## 9. Declaración del uso de herramientas de IA

[POR ESCRIBIR — equipo] Herramientas, finalidad, lo que no se delegó, referencias, contribución de los integrantes y las decisiones técnicas más importantes del avance 5.